# Phase 1: measurement model

Evaluate the 16 scales and 71 questionnaire items before structural analysis. Run all cells in order. The bundled 71-item questionnaire text and its source notes are separate inputs from the response CSV.


## 1. Connect Google Drive

On Colab, mount Drive. On a local system, use local paths below.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Outside Colab: set local paths in the configuration cell.')

## 2. Configuration

Set the input CSV, official 71-item questionnaire, and results directory. All sample sizes are derived from the input.


In [ ]:
# Edit these three paths or set the matching environment variables.
import os
INPUT_CSV_PATH = os.environ.get('PHASE1_INPUT_CSV', '/content/drive/MyDrive/SOICT-2026/data/soict_2026_survey_responses.csv')
QUESTION_TEXT_PATH = os.environ.get('PHASE1_QUESTION_FILE', '/content/drive/MyDrive/SOICT-2026/data/questionnaire.json')
OUTPUT_BASE_DIR = os.environ.get('PHASE1_OUTPUT_DIR', '/content/drive/MyDrive/SOICT-2026/results/phase_01')

# Prespecified analysis settings; record changes when rerunning.
SEED = 20260906
BOOTSTRAP_REPS = 1000
HTMT_REVIEW, HTMT_SERIOUS = 0.85, 0.90
CFI_TARGET, TLI_TARGET, RMSEA_LIMIT, SRMR_LIMIT = 0.95, 0.95, 0.08, 0.08
DELTA_CFI_LIMIT, DELTA_RMSEA_LIMIT = -0.010, 0.015
LATENT_CORR_CRITICAL = 0.95
VCOV_EIGEN_TOLERANCE = -1e-8
COMPARE_NINE_PAIRS = True
RUN_METHOD_SENSITIVITY = True
RUN_POLYCHORIC_DIAGNOSTICS = True
INSTALL_R_ON_COLAB = True

# Optional source correction provenance.
DATA_CORRECTION_PROVENANCE = ''
print('Input:', INPUT_CSV_PATH, '\nQuestionnaire:', QUESTION_TEXT_PATH, '\nOutput:', OUTPUT_BASE_DIR)

## 3. Dependencies and scale definitions


In [ ]:
import importlib.util, subprocess, sys
for module, package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('openpyxl','openpyxl'),('docx','python-docx')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import hashlib, itertools, json, os, platform, re, shutil, unicodedata, warnings
from collections import OrderedDict
from datetime import datetime, timezone
from docx import Document
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value)

SCALE_ITEMS = OrderedDict([
 ('ENP',6),('PTED',6),('MEU',5),('REH',3),('ELA',3),('ORG',3),
 ('CRT',3),('MSR',4),('TSEM',3),('VLS',6),('DEL',7),('LPSN',5),
 ('SFN',4),('TECN',6),('PU',4),('BI',3)])
SCALE_ITEMS = OrderedDict((s,[f'{s}{i:02d}' for i in range(1,k+1)]) for s,k in SCALE_ITEMS.items())
TIERS = {'TIER1':['ENP','PTED','MEU'],
         'TIER2':['REH','ELA','ORG','CRT','MSR','TSEM','VLS','DEL'],
         'TIER3':['LPSN','SFN','TECN'], 'TIER4':['PU','BI']}
ANCHORS = {'NECESSITY':['ENP','LPSN','SFN','TECN'], 'DIFFICULTY':['PTED'],
           'FREQUENCY':['MEU','REH','ELA','ORG','CRT','MSR','TSEM','VLS','DEL'],
           'AGREEMENT':['PU','BI']}
NINE_PAIRS = [('SFN','TECN'),('ELA','ORG'),('LPSN','SFN'),('VLS','DEL'),
              ('REH','ELA'),('ORG','CRT'),('LPSN','TECN'),('PU','BI'),('MSR','TSEM')]
ALL_ITEMS = sum(SCALE_ITEMS.values(), [])
NON_ENP_ITEMS = [x for x in ALL_ITEMS if x not in SCALE_ITEMS['ENP']]
SCALE_TIER = {s:t for t,scales in TIERS.items() for s in scales}
SCALE_ANCHOR = {s:a for a,scales in ANCHORS.items() for s in scales}
ITEM_SCALE = {x:s for s,items in SCALE_ITEMS.items() for x in items}
assert len(SCALE_ITEMS)==16 and len(ALL_ITEMS)==71 and len(set(ALL_ITEMS))==71
display(pd.DataFrame([{'thang':s,'tang':SCALE_TIER[s],'neo':SCALE_ANCHOR[s],'so_cau':len(items)}
                      for s,items in SCALE_ITEMS.items()]))

## 4. Read and validate the input

Validate the 71 responses, the S0 routing rule, the ID field, and file integrity.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def read_csv_auto(path):
    for enc in ['utf-8-sig','utf-8','cp1258','cp1252','latin1']:
        for sep in [';',',','\t']:
            try:
                d = pd.read_csv(path, encoding=enc, sep=sep, dtype=str, keep_default_na=True)
                if set(ALL_ITEMS).issubset(d.columns): return d,enc,sep
            except (UnicodeError, pd.errors.ParserError): pass
    raise ValueError('Cannot read the 71 item columns; check the CSV encoding, delimiter, and headers.')

input_path = Path(INPUT_CSV_PATH).expanduser()
question_path = Path(QUESTION_TEXT_PATH).expanduser()
if not input_path.is_file(): raise FileNotFoundError(f'Input CSV not found: {input_path}')
def locate_directory(directory):
    # Resolve directory names across NFC and NFD Unicode forms.
    cursor=Path(directory.anchor) if directory.is_absolute() else Path.cwd()
    parts=directory.parts[1:] if directory.is_absolute() else directory.parts
    for part in parts:
        direct=cursor/part
        if direct.is_dir():
            cursor=direct
            continue
        if not cursor.is_dir(): return None
        same=[p for p in cursor.iterdir() if p.is_dir() and
              unicodedata.normalize('NFC',p.name).casefold()==unicodedata.normalize('NFC',part).casefold()]
        if len(same)!=1: return None
        cursor=same[0]
    return cursor

if not question_path.is_file():
    folder=locate_directory(question_path.parent)
    if folder is None:
        raise FileNotFoundError(
            f'Questionnaire directory not found: {question_path.parent}. '
            'Mount Google Drive in section 1 and set the correct QUESTION_TEXT_PATH.')
    name_key=unicodedata.normalize('NFC',re.sub(r'\s*\(\d+\)$','',question_path.stem)).casefold()
    accepted={'.xlsx','.xls','.csv','.tsv','.json','.md','.txt','.docx'}
    candidates=[p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in accepted and
                unicodedata.normalize('NFC',re.sub(r'\s*\(\d+\)$','',p.stem)).casefold()==name_key]
    if len(candidates)==1:
        question_path=candidates[0]
        print('The configured question path was missing; matched questionnaire:',question_path)
    else:
        nearby=[p.name for p in folder.iterdir() if p.is_file() and p.suffix.lower() in accepted]
        raise FileNotFoundError(
            f'Questionnaire file not found: {question_path}. '
            f'Similar filenames: {[p.name for p in candidates]}. '
            f'Supported files in the directory: {nearby[:20]}. '
            'Upload the complete official questionnaire and set its actual path in QUESTION_TEXT_PATH.')
output_base = Path(OUTPUT_BASE_DIR).expanduser()
output_base.mkdir(parents=True,exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = output_base / ('phase_01_'+stamp)
run_dir.mkdir(exist_ok=False)
input_sha,question_sha = sha256(input_path),sha256(question_path)
raw,encoding,separator = read_csv_auto(input_path)
if raw.columns.duplicated().any(): raise ValueError('CSV column names must be unique.')
required = ALL_ITEMS+['S0_ENGLISH_NECESSARY']
missing_cols = [x for x in required if x not in raw]
if missing_cols: raise ValueError('Missing column: '+', '.join(missing_cols))
numeric = raw[required].apply(pd.to_numeric,errors='coerce')
bad_string = raw[required].notna() & numeric.isna()
s0 = numeric['S0_ENGLISH_NECESSARY']
enp_missing = numeric[SCALE_ITEMS['ENP']].isna().sum(axis=1)
checks = []
def check(name, ok, detail): checks.append({'kiem_tra':name,'dat':bool(ok),'chi_tiet':str(detail)})
check('Has respondents',len(raw)>0,len(raw))
check('All 71 items defined',len(ALL_ITEMS)==71,'16 thang')
check('All responses are numeric or missing as allowed', not bad_string.any().any(),int(bad_string.sum().sum()))
check('S0 is observed and binary',s0.isin([0,1]).all(),s0.value_counts(dropna=False).to_dict())
check('ENP follows the survey routing rule',((s0.eq(0)&enp_missing.eq(6))|(s0.eq(1)&enp_missing.eq(0))).all(),
      f'{int((s0==0).sum())} not applicable; {int((s0==1).sum())} applicable')
check('The 65 other items are complete',numeric[NON_ENP_ITEMS].notna().all().all(),
      int(numeric[NON_ENP_ITEMS].isna().sum().sum()))
check('Integer responses from 1 to 5',all((numeric[x].dropna().isin([1,2,3,4,5])).all() for x in ALL_ITEMS),
      'Allowed values are {1,2,3,4,5}')
if 'RESPONDENT_ID' in raw:
    check('Respondent IDs are observed and unique',raw.RESPONDENT_ID.notna().all() and
          raw.RESPONDENT_ID.is_unique and raw.RESPONDENT_ID.astype(str).str.strip().ne('').all(),
          f'repeated IDs: {int(raw.RESPONDENT_ID.duplicated().sum())}')
else: check('Respondent ID',True,'No ID column supplied; use row-position IDs for this run')
checks_df = pd.DataFrame(checks)
checks_df.to_csv(run_dir/'01_kiem_tra_dau_vao.csv',index=False,encoding='utf-8-sig')
display(checks_df)
if not checks_df.dat.all(): raise ValueError('Input validation failed; inspect 01_kiem_tra_dau_vao.csv')

df = numeric[ALL_ITEMS+['S0_ENGLISH_NECESSARY']].copy()
df.insert(0,'RESPONDENT_ID',raw.RESPONDENT_ID.to_numpy() if 'RESPONDENT_ID' in raw else
          [f'ROW{i:06d}' for i in range(1,len(raw)+1)])
primary = df.loc[df[ALL_ITEMS].notna().all(axis=1)].copy()
full65 = df[NON_ENP_ITEMS].copy()
counts = {'n':len(df),'n_71':len(primary),'n_65':len(full65),'n_enp_structural_missing':int(s0.eq(0).sum())}
pd.DataFrame([counts]).to_csv(run_dir/'02_co_mau_va_phan_nhanh.csv',index=False,encoding='utf-8-sig')
df.to_csv(run_dir/'00_du_lieu_phan_tich_khong_sua_nguon.csv',index=False,encoding='utf-8')
display(pd.DataFrame([counts]))

## 5. Load the complete 71-item questionnaire

The questionnaire can be CSV, TSV, XLSX, JSON, Markdown, TXT, or DOCX. All 71 item texts must be present.


In [ ]:
from openpyxl import load_workbook

# Read all item texts from the configured questionnaire source.
TECN_OFFICIAL = {}

def normalize_col(s):
    s=unicodedata.normalize('NFD',str(s).lower())
    return re.sub(r'[^a-z0-9]','', ''.join(ch for ch in s if not unicodedata.combining(ch)))

def question_records(path):
    suffix=path.suffix.lower()
    if suffix in ('.csv','.tsv','.xlsx','.xls'):
        table = (pd.read_excel(path,dtype=str) if suffix in ('.xlsx','.xls') else
                 pd.read_csv(path,sep='\t' if suffix=='.tsv' else None,engine='python',dtype=str,
                             encoding='utf-8-sig'))
        normalized={normalize_col(c):c for c in table.columns}
        item_col=next((normalized[x] for x in ['item','macau','maitem','code','itemcode'] if x in normalized),None)
        text_col=next((normalized[x] for x in ['text','noidung','noidungcau','question','cauhoi'] if x in normalized),None)
        if item_col is not None and text_col is not None:
            return [(str(row[item_col]).strip().upper(),str(row[text_col]).strip()) for _,row in table.iterrows()]
        if suffix not in ('.xlsx','.xls'):
            raise ValueError('The questionnaire CSV/TSV requires an item code and question-text column.')
        # Google Forms export: find item codes and complete text in column headers.
        records=[]
        for sheet in load_workbook(path,read_only=True,data_only=True).worksheets:
            for row in sheet.values:
                for value in row:
                    if not isinstance(value,str): continue
                    match=re.search(r'\[\s*([A-Z]{2,5}\d{2})\s*[.。:]\s*(.*?)\s*\]',value,re.S)
                    if match:
                        records.append((match.group(1),match.group(2).strip()))
        return records
    if suffix=='.json':
        obj=json.loads(path.read_text(encoding='utf-8-sig'))
        if isinstance(obj,dict): return [(str(k).strip().upper(),str(v).strip()) for k,v in obj.items()]
        if isinstance(obj,list):
            return [(str(d.get('item',d.get('ma_cau',''))).strip().upper(),
                     str(d.get('text',d.get('noi_dung',''))).strip()) for d in obj]
        raise ValueError('JSON must map item codes to text or contain item/text records.')
    if suffix=='.docx':
        doc=Document(path)
        content='\n'.join([p.text for p in doc.paragraphs]+['\n'.join(' | '.join(c.text for c in row.cells)
                                                        for row in t.rows) for t in doc.tables])
    elif suffix in ('.md','.txt'):
        content=path.read_text(encoding='utf-8-sig')
    else: raise ValueError('Unsupported questionnaire format; use CSV, TSV, XLSX, JSON, MD, TXT, or DOCX.')
    pattern=re.compile(r'(?im)^\s*(?:[-*]\s*)?(?:\*\*)?\b([A-Z]{2,5}\d{2})\b(?:\*\*)?\s*[.:)\-–]\s*')
    hits=list(pattern.finditer(content));records=[]
    for i,hit in enumerate(hits):
        body=content[hit.end():hits[i+1].start() if i+1<len(hits) else len(content)]
        body=re.sub(r'\s+',' ',body).strip().strip('* ')
        records.append((hit.group(1).upper(),body))
    return records

records=question_records(question_path)
found={};duplicates=[]
for key,value in records:
    if key in ALL_ITEMS:
        if key in found: duplicates.append(key)
        found[key]=value
# Require complete source wording for every item.
tecn_source = {key: str(question_path) for key in found if key.startswith('TECN')}
TECN_OFFICIAL = {key: found[key] for key in SCALE_ITEMS['TECN'] if key in found}
missing=[x for x in ALL_ITEMS if not found.get(x) or found[x].lower() in ('nan','none')]
if duplicates or missing:
    raise ValueError(f'The official question file is incomplete: repeated item codes {duplicates}; missing item codes {missing}.')
questions=pd.DataFrame([{'ma_cau':x,'thang':ITEM_SCALE[x],'tang':SCALE_TIER[ITEM_SCALE[x]],
                         'neo':SCALE_ANCHOR[ITEM_SCALE[x]],'noi_dung_chinh_thuc':found[x],
                         'nguon_noi_dung':tecn_source.get(x,str(question_path))}
                        for x in ALL_ITEMS])
questions.to_csv(run_dir/'03_noi_dung_71_cau.csv',index=False,encoding='utf-8-sig')
pair_content=[]
for a,b in NINE_PAIRS:
    pair_content.append({'thang_1':a,'thang_2':b,
        'cau_thang_1':' | '.join(found[x] for x in SCALE_ITEMS[a]),
        'cau_thang_2':' | '.join(found[x] for x in SCALE_ITEMS[b]),
        'ket_luan_noi_dung':'Examine conceptual boundaries before combining scales'})
pd.DataFrame(pair_content).to_csv(run_dir/'03b_doi_chieu_noi_dung_chin_cap.csv',index=False,encoding='utf-8-sig')
for code in SCALE_ITEMS['TECN']:
    print(code+': '+found[code])
print('Verified complete text for',len(questions),'items.')

## 6. Descriptive statistics and discriminant validity

Estimate Spearman HTMT for all scale pairs and bootstrap the nine prespecified comparisons.


In [ ]:
rng=np.random.default_rng(SEED)
dist=[]
for x in ALL_ITEMS:
    v=df[x].dropna()
    counts_5=v.value_counts().reindex(range(1,6),fill_value=0)
    for k,n_k in counts_5.items():
        dist.append({'ma_cau':x,'thang':ITEM_SCALE[x],'muc':k,'so_nguoi':int(n_k),
                     'ty_le':float(n_k/len(v)) if len(v) else np.nan,
                     'canh_bao_muc_hiem':bool(n_k<5),'o_bang_khong':bool(n_k==0)})
pd.DataFrame(dist).to_csv(run_dir/'04_phan_bo_5_muc.csv',index=False,encoding='utf-8-sig')
scale_summary=[]
for scale,items in SCALE_ITEMS.items():
    block=df[items].dropna();corr=block.corr(method='spearman').to_numpy()
    upper=corr[np.triu_indices(len(items),1)]
    avg=float(np.nanmean(upper))
    alpha=float(len(items)*avg/(1+(len(items)-1)*avg)) if np.isfinite(avg) and 1+(len(items)-1)*avg!=0 else np.nan
    scale_summary.append({'thang':scale,'n_du':len(block),'so_cau':len(items),
        'tuong_quan_spearman_trung_binh':avg,'alpha_dua_tren_tuong_quan_spearman':alpha})
pd.DataFrame(scale_summary).to_csv(run_dir/'05_mo_ta_thang.csv',index=False,encoding='utf-8-sig')

# HTMT: mean absolute between-scale correlation divided by within-scale correlations.
# Return NaN when constant responses make a correlation undefined.
def htmt_spearman(frame,a,b):
    x,y=SCALE_ITEMS[a],SCALE_ITEMS[b]
    corr=frame[x+y].corr(method='spearman')
    cross=corr.loc[x,y].abs().to_numpy().ravel()
    aa=corr.loc[x,x].abs().to_numpy()[np.triu_indices(len(x),1)]
    bb=corr.loc[y,y].abs().to_numpy()[np.triu_indices(len(y),1)]
    if not (np.isfinite(cross).all() and np.isfinite(aa).all() and np.isfinite(bb).all()): return np.nan
    denom=np.sqrt(aa.mean()*bb.mean())
    return float(cross.mean()/denom) if denom>0 else np.nan

pair_rows=[]
for a,b in itertools.combinations(SCALE_ITEMS,2):
    value=htmt_spearman(df,a,b)
    pair_rows.append({'thang_1':a,'thang_2':b,'HTMT_spearman':value,
                      'cung_neo':SCALE_ANCHOR[a]==SCALE_ANCHOR[b],
                      'cung_tang':SCALE_TIER[a]==SCALE_TIER[b],
                      'canh_bao_085':bool(value>=HTMT_REVIEW),
                      'canh_bao_090':bool(value>=HTMT_SERIOUS)})
htmt_all=pd.DataFrame(pair_rows)
htmt_all.to_csv(run_dir/'06_HTMT_120_cap.csv',index=False,encoding='utf-8-sig')

bootstrap=[]
for a,b in NINE_PAIRS:
    # Bootstrap respondents with complete responses on both scales.
    block=df[SCALE_ITEMS[a]+SCALE_ITEMS[b]].dropna().reset_index(drop=True)
    values=np.array([htmt_spearman(block.iloc[rng.integers(0,len(block),len(block))],a,b)
                     for _ in range(BOOTSTRAP_REPS)])
    good=values[np.isfinite(values)]
    bootstrap.append({'thang_1':a,'thang_2':b,'n':len(block),
        'HTMT_spearman':htmt_spearman(block,a,b), 'bootstrap_hop_le':len(good),
        'CI_95_duoi':float(np.quantile(good,.025)) if len(good)>10 else np.nan,
        'CI_95_tren':float(np.quantile(good,.975)) if len(good)>10 else np.nan})
bootstrap_df=pd.DataFrame(bootstrap)
bootstrap_df.to_csv(run_dir/'07_HTMT_bootstrap_chin_cap.csv',index=False,encoding='utf-8-sig')
display(bootstrap_df)

## 7. Check R and lavaan

Run ordered categorical CFA in R/lavaan; do not substitute approximate output when R is unavailable.


In [ ]:
def install_r_colab():
    if shutil.which('Rscript') and subprocess.run(['Rscript','-e',
        "quit(status=ifelse(requireNamespace('lavaan',quietly=TRUE),0,1))"],capture_output=True).returncode==0:
        return
    if not ON_COLAB or not INSTALL_R_ON_COLAB:
        raise RuntimeError('Rscript or lavaan is missing; install both before fitting CFA.')
    commands=[['apt-get','update','-qq'],['apt-get','install','-y','-qq','r-base','r-cran-lavaan']]
    log=run_dir/'00_nhat_ky_cai_R_lavaan.txt'
    with log.open('w',encoding='utf-8') as stream:
        for cmd in commands:
            p=subprocess.run(cmd,stdout=stream,stderr=subprocess.STDOUT)
            if p.returncode: raise RuntimeError(f'R/lavaan installation failed; inspect {log}')
    if subprocess.run(['Rscript','-e',"quit(status=ifelse(requireNamespace('lavaan',quietly=TRUE),0,1))"],
                      capture_output=True).returncode:
        raise RuntimeError('R is present but lavaan is unavailable; inspect the installation log.')

install_r_colab()
r_version=subprocess.run(['Rscript','-e','cat(as.character(getRversion()));cat(" | lavaan ");cat(as.character(packageVersion("lavaan")))'],
                         capture_output=True,text=True,check=True).stdout
print('CFA environment:',r_version)

## 8. Specify the CFA models

Retain the predefined model and comparison set.


In [ ]:
specs=OrderedDict()
def measurement(scales):
    return '\n'.join(f'{s} =~ '+ ' + '.join(SCALE_ITEMS[s]) for s in scales)
def add(name,syntax,items,group):
    if name in specs: raise ValueError(f'Model names must be unique: {name}')
    specs[name]={'syntax':syntax,'items':list(items),'group':group,
                 'sample':'primary' if any(x.startswith('ENP') for x in items) else 'all'}
base=measurement(SCALE_ITEMS)
strategy='STRATEGY =~ '+' + '.join(TIERS['TIER2'])
need='FUNCTIONAL_NEEDS =~ '+' + '.join(TIERS['TIER3'])
add('M1',base,ALL_ITEMS,'six_models')
add('M2',measurement([s for s in SCALE_ITEMS if s!='ENP']),NON_ENP_ITEMS,'six_models')
add('M3',base+'\n'+strategy,ALL_ITEMS,'six_models')
add('M4',base+'\n'+need,ALL_ITEMS,'six_models')
add('M5',base+'\n'+strategy+'\n'+need,ALL_ITEMS,'six_models')
add('M6',base+'\nSTRATEGY_7 =~ '+' + '.join([s for s in TIERS['TIER2'] if s!='DEL']),ALL_ITEMS,'six_models')
for tier,scales in TIERS.items():
    items=sum([SCALE_ITEMS[s] for s in scales],[])
    add(tier+'_CORRELATED',measurement(scales),items,'tier')
add('T2_CLS_SECOND_ORDER',measurement(TIERS['TIER2'])+'\nCLS =~ REH + ELA + ORG + CRT',
    sum([SCALE_ITEMS[s] for s in TIERS['TIER2']],[]),'competing')
add('T2_STRATEGY_SECOND_ORDER',measurement(TIERS['TIER2'])+'\n'+strategy,
    sum([SCALE_ITEMS[s] for s in TIERS['TIER2']],[]),'competing')
add('T2_ONE_FACTOR_32','T2_ONE =~ '+' + '.join(sum([SCALE_ITEMS[s] for s in TIERS['TIER2']],[])),
    sum([SCALE_ITEMS[s] for s in TIERS['TIER2']],[]),'competing')
add('T2_FOUR_COMPONENTS',measurement(['VLS','DEL'])+'\nCLS_ITEMS =~ '+
    ' + '.join(sum([SCALE_ITEMS[s] for s in ['REH','ELA','ORG','CRT']],[]))+
    '\nSRL_ITEMS =~ '+' + '.join(sum([SCALE_ITEMS[s] for s in ['MSR','TSEM']],[])),
    sum([SCALE_ITEMS[s] for s in TIERS['TIER2']],[]),'competing')
add('T3_NEED_SECOND_ORDER',measurement(TIERS['TIER3'])+'\n'+need,
    sum([SCALE_ITEMS[s] for s in TIERS['TIER3']],[]),'competing')
add('T3_ONE_FACTOR_15','T3_ONE =~ '+' + '.join(sum([SCALE_ITEMS[s] for s in TIERS['TIER3']],[])),
    sum([SCALE_ITEMS[s] for s in TIERS['TIER3']],[]),'competing')
add('T3_MERGE_SFN_TECN',measurement(['LPSN'])+'\nSFN_TECN =~ '+
    ' + '.join(SCALE_ITEMS['SFN']+SCALE_ITEMS['TECN']),
    sum([SCALE_ITEMS[s] for s in TIERS['TIER3']],[]),'competing')
add('T4_ONE_FACTOR_7','T4_ONE =~ '+' + '.join(sum([SCALE_ITEMS[s] for s in TIERS['TIER4']],[])),
    sum([SCALE_ITEMS[s] for s in TIERS['TIER4']],[]),'competing')
add('FULL_CLS_SECOND_ORDER',base+'\nCLS =~ REH + ELA + ORG + CRT',ALL_ITEMS,'competing')
add('FULL_CLS_NEED_SECOND_ORDER',base+'\nCLS =~ REH + ELA + ORG + CRT\n'+need,ALL_ITEMS,'competing')
ela_org_scales=[s for s in SCALE_ITEMS if s not in ('ELA','ORG')]
add('FULL_MERGE_ELA_ORG',measurement(ela_org_scales)+'\nELA_ORG =~ '+
    ' + '.join(SCALE_ITEMS['ELA']+SCALE_ITEMS['ORG']),ALL_ITEMS,'competing')
sfn_tecn_scales=[s for s in SCALE_ITEMS if s not in ('SFN','TECN')]
add('FULL_MERGE_SFN_TECN',measurement(sfn_tecn_scales)+'\nSFN_TECN =~ '+
    ' + '.join(SCALE_ITEMS['SFN']+SCALE_ITEMS['TECN']),ALL_ITEMS,'competing')
add('FULL_STRATEGY_SECOND_ORDER',base+'\n'+strategy,ALL_ITEMS,'confirmatory')
add('FULL_NEED_SECOND_ORDER',base+'\n'+need,ALL_ITEMS,'confirmatory')
add('FULL_STRATEGY_NEED_SECOND_ORDER',base+'\n'+strategy+'\n'+need,ALL_ITEMS,'confirmatory')
add('FULL_16_CORRELATED',base,ALL_ITEMS,'confirmatory')
add('FULL_ONE_FACTOR_71','ONE_FACTOR =~ '+' + '.join(ALL_ITEMS),ALL_ITEMS,'method_sensitivity')
add('FULL_ONE_FACTOR_65','ONE_FACTOR =~ '+' + '.join(NON_ENP_ITEMS),NON_ENP_ITEMS,'method_sensitivity')
for a,b in NINE_PAIRS if COMPARE_NINE_PAIRS else []:
    items=SCALE_ITEMS[a]+SCALE_ITEMS[b]
    add('PAIR_'+a+'_'+b+'_TWO',measurement([a,b]),items,'pair')
    add('PAIR_'+a+'_'+b+'_ONE',a+'_'+b+'_ONE =~ '+' + '.join(items),items,'pair')

# Method-factor sensitivity separates response-format and anchor variation.
if RUN_METHOD_SENSITIVITY:
    add('FULL_GLOBAL_METHOD',base+'\nMETHOD =~ '+' + '.join(ALL_ITEMS)+
        '\nMETHOD ~~ 0*'+' + 0*'.join(SCALE_ITEMS.keys()),ALL_ITEMS,'method_sensitivity')
    freq=sum([SCALE_ITEMS[s] for s in ANCHORS['FREQUENCY']],[])
    necessity=sum([SCALE_ITEMS[s] for s in ANCHORS['NECESSITY']],[])
    method_lines=['ANCHOR_FREQ =~ '+' + '.join(freq),'ANCHOR_NECESSITY =~ '+' + '.join(necessity),
                  'ANCHOR_FREQ ~~ 0*ANCHOR_NECESSITY']
    for method in ('ANCHOR_FREQ','ANCHOR_NECESSITY'):
        method_lines.append(method+' ~~ 0*'+' + 0*'.join(SCALE_ITEMS.keys()))
    add('FULL_ANCHOR_METHODS',base+'\n'+'\n'.join(method_lines),ALL_ITEMS,'method_sensitivity')

model_table=pd.DataFrame([{'model':name,'group':x['group'],'sample':x['sample'],
                           'n_items':len(x['items']),'syntax':x['syntax']} for name,x in specs.items()])
model_table.to_csv(run_dir/'08_dac_ta_toan_bo_mo_hinh.csv',index=False,encoding='utf-8-sig')
display(model_table[['model','group','sample','n_items']])

## 9. Fit CFA and inspect diagnostics


In [ ]:
# Transfer only the required 71 response columns to R.
prepared=run_dir/'00_du_lieu_71_cau_cho_R.csv'
df[ALL_ITEMS].to_csv(prepared,index=False,encoding='utf-8')
def r_string(s): return json.dumps(str(s),ensure_ascii=False)
def r_vector(xs): return 'c('+','.join(r_string(x) for x in xs)+')'
model_lines=[]
for name,entry in specs.items():
    model_lines.append(f'{name}=list(syntax={r_string(entry["syntax"])},items={r_vector(entry["items"])},'
                       f'group={r_string(entry["group"])},sample={r_string(entry["sample"])})')
models_r='list(\n'+',\n'.join(model_lines)+'\n)'
r_template=r'''
suppressPackageStartupMessages(library(lavaan))
args <- commandArgs(trailingOnly=TRUE)
input <- args[[1]]; out <- args[[2]]; do_poly <- args[[3]] == "TRUE"
models <- __MODELS__
dat <- read.csv(input, check.names=FALSE, na.strings=c("", "NA"), fileEncoding="UTF-8")
write_out <- function(x,name) write.csv(x,file.path(out,name),row.names=FALSE,na="",fileEncoding="UTF-8")
rowbind_safe <- function(xs) {
  if(!length(xs)) return(data.frame())
  all_names <- unique(unlist(lapply(xs,names),use.names=FALSE))
  aligned <- lapply(xs,function(x){
    for(nm in setdiff(all_names,names(x))) x[[nm]] <- rep(NA,nrow(x))
    x[,all_names,drop=FALSE]
  })
  do.call(rbind,aligned)
}
fm <- function(fit,key) tryCatch(as.numeric(fitMeasures(fit,key)[1]),error=function(e) NA_real_)
fits <- list(); all_params <- list(); all_r2 <- list(); all_mi <- list(); all_residual <- list(); all_cor <- list()
fit_cache <- list()
for (name in names(models)) {
  x <- models[[name]]
  block <- dat[,x$items,drop=FALSE]
  if (x$sample == "primary") block <- block[complete.cases(block),,drop=FALSE]
  # lavaan WLSMV treats Likert items as ordered and applies listwise missingness.
  warnings_seen <- character(0)
  key <- paste(c(x$syntax,x$items,x$sample),collapse="|")
  if (key %in% names(fit_cache)) {
    fit <- fit_cache[[key]]
  } else {
    fit <- tryCatch(withCallingHandlers(
        cfa(model=x$syntax,data=block,ordered=x$items,estimator="WLSMV",std.lv=TRUE,
            parameterization="theta",missing="listwise",control=list(iter.max=3000)),
        warning=function(w){warnings_seen <<- c(warnings_seen,conditionMessage(w));invokeRestart("muffleWarning")}),
        error=function(e){warnings_seen <<- c(warnings_seen,conditionMessage(e));NULL})
    if (!is.null(fit)) fit_cache[[key]] <- fit
  }
  ok <- !is.null(fit) && isTRUE(tryCatch(lavInspect(fit,"converged"),error=function(e) FALSE))
  if (ok) {
    post_check <- isTRUE(tryCatch(lavInspect(fit,"post.check"),error=function(e) FALSE))
    parameter <- tryCatch(parameterEstimates(fit,standardized=TRUE),error=function(e) NULL)
    if (!is.null(parameter) && nrow(parameter)>0) {
      parameter$model <- name;all_params[[name]] <- parameter
    }
    r2 <- tryCatch(lavInspect(fit,"rsquare"),error=function(e) NULL)
    if(!is.null(r2) && length(r2)>0) {
      r2_names <- names(r2)
      if(is.null(r2_names)) r2_names <- as.character(seq_along(r2))
      all_r2[[name]] <- data.frame(model=name,item=r2_names,r2=as.numeric(r2))
    }
    mi <- tryCatch(modificationIndices(fit,sort.=TRUE),error=function(e) NULL)
    if(!is.null(mi) && nrow(mi)>0) {mi$model <- name;all_mi[[name]] <- mi}
    corlv <- tryCatch(lavInspect(fit,"cor.lv"),error=function(e) NULL)
    if(!is.null(corlv) && is.matrix(corlv) && nrow(corlv)>1) {
      ij <- which(upper.tri(corlv),arr.ind=TRUE)
      if(nrow(ij)>0) {
        rnames <- rownames(corlv);cnames <- colnames(corlv)
        if(is.null(rnames)) rnames <- as.character(seq_len(nrow(corlv)))
        if(is.null(cnames)) cnames <- as.character(seq_len(ncol(corlv)))
        all_cor[[name]] <- data.frame(model=name,latent_1=rnames[ij[,1]],
          latent_2=cnames[ij[,2]],correlation=as.numeric(corlv[ij]))
      }
    }
    res <- tryCatch(residuals(fit,type="cor")$cov,error=function(e) NULL)
    if(!is.null(res) && is.matrix(res) && nrow(res)>1) {
      ij <- which(upper.tri(res),arr.ind=TRUE)
      if(nrow(ij)>0) {
        rnames <- rownames(res);cnames <- colnames(res)
        if(is.null(rnames)) rnames <- as.character(seq_len(nrow(res)))
        if(is.null(cnames)) cnames <- as.character(seq_len(ncol(res)))
        all_residual[[name]] <- data.frame(model=name,item_1=rnames[ij[,1]],
          item_2=cnames[ij[,2]],residual=as.numeric(res[ij]))
      }
    }
    psi <- tryCatch(diag(lavInspect(fit,"cov.lv")),error=function(e) NA_real_)
    min_latent_variance <- suppressWarnings(min(psi,na.rm=TRUE))
    if(!is.finite(min_latent_variance)) min_latent_variance <- NA_real_
    bad_se <- if(is.null(parameter)) NA_integer_ else sum(!is.finite(parameter$se),na.rm=TRUE)
    negative_variances <- if(is.null(parameter)) NA_integer_ else sum(
      parameter$op=="~~" & parameter$lhs==parameter$rhs & parameter$est < -1e-6,na.rm=TRUE)
    covariance_eigen_min <- NA_real_
    if (x$group %in% c("six_models","confirmatory")) {
      vcm <- tryCatch(vcov(fit),error=function(e) NULL)
      if (!is.null(vcm) && nrow(vcm)>0) covariance_eigen_min <- tryCatch(
        min(eigen((vcm+t(vcm))/2,symmetric=TRUE,only.values=TRUE)$values),
        error=function(e) NA_real_)
    }
    npar <- tryCatch(as.integer(lavInspect(fit,"npar")),error=function(e) NA_integer_)
  } else {min_latent_variance <- NA_real_;bad_se <- NA_integer_;npar <- NA_integer_;
          post_check <- FALSE;negative_variances <- NA_integer_;covariance_eigen_min <- NA_real_}
  fits[[name]] <- data.frame(model=name,group=x$group,sample=x$sample,
    n=nrow(block),converged=ok,post_check=post_check,npar=npar,
    chisq_scaled=if(ok) fm(fit,"chisq.scaled") else NA_real_,
    df_scaled=if(ok) fm(fit,"df.scaled") else NA_real_,
    p_scaled=if(ok) fm(fit,"pvalue.scaled") else NA_real_,
    cfi_scaled=if(ok) fm(fit,"cfi.scaled") else NA_real_,
    tli_scaled=if(ok) fm(fit,"tli.scaled") else NA_real_,
    rmsea_scaled=if(ok) fm(fit,"rmsea.scaled") else NA_real_,
    srmr=if(ok) fm(fit,"srmr") else NA_real_,
    cfi_robust=if(ok) fm(fit,"cfi.robust") else NA_real_,
    tli_robust=if(ok) fm(fit,"tli.robust") else NA_real_,
    rmsea_robust=if(ok) fm(fit,"rmsea.robust") else NA_real_,
    min_latent_variance=min_latent_variance,bad_se=bad_se,
    negative_variances=negative_variances,vcov_eigen_min=covariance_eigen_min,
    warnings=paste(unique(warnings_seen),collapse=" | "),stringsAsFactors=FALSE)
  cat(name, " n=",nrow(block)," converged=",ok," warnings=",length(warnings_seen),"\n",sep="")
  flush.console()
}
write_out(rowbind_safe(fits),"09_chi_so_CFA_tat_ca_mo_hinh.csv")
write_out(rowbind_safe(all_params),"10_tat_ca_tham_so_CFA.csv")
write_out(rowbind_safe(all_r2),"11_R2_cau_hoi.csv")
write_out(rowbind_safe(all_mi),"12_chi_so_dieu_chinh_MI.csv")
write_out(rowbind_safe(all_residual),"13_phan_du_CFA.csv")
write_out(rowbind_safe(all_cor),"14_tuong_quan_nhan_to.csv")

if (do_poly) {
  complete71 <- dat[complete.cases(dat),,drop=FALSE]
  items65 <- names(dat)[!names(dat)%in%c("ENP01","ENP02","ENP03","ENP04","ENP05","ENP06")]
  blocks <- list(FULL_71=complete71,
                 FULL_65_ON_SAME_PEOPLE=complete71[,items65,drop=FALSE],
                 FULL_65_ALL=dat[,items65,drop=FALSE])
  for (tier in c("TIER1","TIER2","TIER3","TIER4")) {
    scope <- names(models)[names(models)==paste0(tier,"_CORRELATED")]
    if(length(scope)) {
      vars <- models[[scope]]$items
      blocks[[tier]] <- dat[,vars,drop=FALSE]
      blocks[[tier]] <- blocks[[tier]][complete.cases(blocks[[tier]]),,drop=FALSE]
    }
  }
  poly_rows <- list(); eigen_rows <- list(); poly_errors <- list()
  for(name in names(blocks)) {
    x <- blocks[[name]]
    warning_text <- character(0)
    cor <- tryCatch(withCallingHandlers(lavCor(x,ordered=names(x),output="cor"),
      warning=function(w){warning_text <<- c(warning_text,conditionMessage(w));invokeRestart("muffleWarning")}),
      error=function(e){warning_text <<- c(warning_text,conditionMessage(e));NULL})
    if(is.null(cor)) {
      poly_errors[[name]] <- data.frame(block=name,n=nrow(x),error=paste(warning_text,collapse=" | "))
      next
    }
    ij <- which(upper.tri(cor),arr.ind=TRUE)
    if(nrow(ij)>0) {
      rnames <- rownames(cor);cnames <- colnames(cor)
      if(is.null(rnames)) rnames <- as.character(seq_len(nrow(cor)))
      if(is.null(cnames)) cnames <- as.character(seq_len(ncol(cor)))
      poly_rows[[name]] <- data.frame(block=name,item_1=rnames[ij[,1]],
        item_2=cnames[ij[,2]],polychoric=as.numeric(cor[ij]))
    }
    ev <- tryCatch(eigen((cor+t(cor))/2,symmetric=TRUE,only.values=TRUE)$values,
                   error=function(e){warning_text <<- c(warning_text,conditionMessage(e));numeric(0)})
    if(length(ev)>0) eigen_rows[[name]] <- data.frame(block=name,index=seq_along(ev),eigenvalue=as.numeric(ev))
    poly_errors[[name]] <- data.frame(block=name,n=nrow(x),error=paste(unique(warning_text),collapse=" | "))
  }
  write_out(rowbind_safe(poly_rows),"15_ma_tran_polychoric_cap_cau.csv")
  write_out(rowbind_safe(eigen_rows),"16_tri_rieng_polychoric.csv")
  write_out(rowbind_safe(poly_errors),"17_nhat_ky_polychoric.csv")
}
'''
r_script=r_template.replace('__MODELS__',models_r)
r_path=run_dir/'00_chay_CFA_WLSMV_lavaan.R'
r_path.write_text(r_script,encoding='utf-8')
log_path=run_dir/'00_nhat_ky_R_lavaan.txt'
cmd=['Rscript',str(r_path),str(prepared),str(run_dir),str(bool(RUN_POLYCHORIC_DIAGNOSTICS)).upper()]
with log_path.open('w',encoding='utf-8') as log:
    proc=subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT)
if proc.returncode:
    tail=log_path.read_text(encoding='utf-8',errors='replace')[-10000:]
    raise RuntimeError(
        f'R/lavaan CFA exited with code {proc.returncode}. Log: {log_path}\n'
        f'--- Last 10,000 characters from the R log ---\n{tail or "R log is empty"}')
print('CFA completed. Log:',log_path)

## 10. Compare models with prespecified criteria

Interpret fit, parameter admissibility, discriminant evidence, and sensitivity together.


In [ ]:
fits=pd.read_csv(run_dir/'09_chi_so_CFA_tat_ca_mo_hinh.csv',encoding='utf-8-sig')
if not fits.converged.fillna(False).any():
    raise RuntimeError('No CFA model converged; inspect the R log and model fit CSV.')
params=pd.read_csv(run_dir/'10_tat_ca_tham_so_CFA.csv',encoding='utf-8-sig')
cor=pd.read_csv(run_dir/'14_tuong_quan_nhan_to.csv',encoding='utf-8-sig')
baseline=fits.set_index('model').loc['FULL_16_CORRELATED']
def finite(v): return pd.notna(v) and np.isfinite(float(v))
def valid_row(row):
    return (bool(row['converged']) and bool(row['post_check']) and finite(row['min_latent_variance']) and
            row['min_latent_variance']>=0 and finite(row['bad_se']) and int(row['bad_se'])==0 and
            finite(row['negative_variances']) and row['negative_variances']==0 and
            (not finite(row['vcov_eigen_min']) or row['vcov_eigen_min']>=VCOV_EIGEN_TOLERANCE))

comp=[]
for _,row in fits.iterrows():
    comparable=row['sample']=='primary' and row['n']==baseline['n'] and set(specs[row['model']]['items'])==set(ALL_ITEMS)
    dcf=float(row.cfi_scaled-baseline.cfi_scaled) if comparable and finite(row.cfi_scaled) and finite(baseline.cfi_scaled) else np.nan
    drm=float(row.rmsea_scaled-baseline.rmsea_scaled) if comparable and finite(row.rmsea_scaled) and finite(baseline.rmsea_scaled) else np.nan
    absolute=valid_row(row) and all(finite(row[x]) for x in ['cfi_scaled','tli_scaled','rmsea_scaled','srmr']) and (
        row.cfi_scaled>=CFI_TARGET and row.tli_scaled>=TLI_TARGET and row.rmsea_scaled<=RMSEA_LIMIT and row.srmr<=SRMR_LIMIT)
    relative=comparable and finite(dcf) and finite(drm) and dcf>=DELTA_CFI_LIMIT and drm<=DELTA_RMSEA_LIMIT
    comp.append({'model':row['model'],'group':row['group'],'n':row['n'],'converged':row['converged'],
        'admissible':valid_row(row),'cfi_scaled':row.cfi_scaled,'tli_scaled':row.tli_scaled,
        'rmsea_scaled':row.rmsea_scaled,'srmr':row.srmr,'delta_cfi_vs_full16':dcf,
        'delta_rmsea_vs_full16':drm,'absolute_fit_pass':bool(absolute),
        'relative_pass':bool(relative) if comparable else np.nan})
comparison=pd.DataFrame(comp)
comparison.to_csv(run_dir/'18_so_sanh_va_hau_kiem_mo_hinh.csv',index=False,encoding='utf-8-sig')

pair_decisions=[]
for a,b in NINE_PAIRS:
    ra=fits.set_index('model').loc['PAIR_'+a+'_'+b+'_TWO'] if COMPARE_NINE_PAIRS else None
    rb=fits.set_index('model').loc['PAIR_'+a+'_'+b+'_ONE'] if COMPARE_NINE_PAIRS else None
    h=bootstrap_df.loc[(bootstrap_df.thang_1==a)&(bootstrap_df.thang_2==b)].iloc[0]
    cpair=cor.loc[cor.model.eq('PAIR_'+a+'_'+b+'_TWO')] if COMPARE_NINE_PAIRS else pd.DataFrame()
    latent_corr=float(cpair.correlation.iloc[0]) if len(cpair) else np.nan
    dcfi=float(ra.cfi_scaled-rb.cfi_scaled) if ra is not None and finite(ra.cfi_scaled) and finite(rb.cfi_scaled) else np.nan
    drm=float(ra.rmsea_scaled-rb.rmsea_scaled) if ra is not None and finite(ra.rmsea_scaled) and finite(rb.rmsea_scaled) else np.nan
    better=finite(dcfi) and finite(drm) and (dcfi>=.010 or drm<=-.015)
    available=ra is not None and valid_row(ra) and valid_row(rb) and finite(h.HTMT_spearman) and finite(h.CI_95_tren) and finite(latent_corr)
    strong=available and h.HTMT_spearman<.85 and h.CI_95_tren<.90 and abs(latent_corr)<.90 and better
    tentative=available and h.HTMT_spearman<.90 and abs(latent_corr)<.95 and better
    verdict=('strong evidence of scale separation' if strong else
             'conditional evidence of scale separation' if tentative else
             'insufficient evidence of scale separation' if available else 'indeterminate')
    pair_decisions.append({'thang_1':a,'thang_2':b,'htmt':h.HTMT_spearman,'htmt_ci_upper':h.CI_95_tren,
                           'latent_corr':latent_corr,'delta_cfi_two_minus_one':dcfi,
                           'delta_rmsea_two_minus_one':drm,'ket_luan_thong_ke':verdict,
                           'yeu_cau_doi_chieu_noi_dung':True})
pd.DataFrame(pair_decisions).to_csv(run_dir/'19_bang_chung_chin_cap.csv',index=False,encoding='utf-8-sig')

target=comparison.set_index('model').loc['FULL_STRATEGY_NEED_SECOND_ORDER']
high=params.loc[params.model.eq('FULL_STRATEGY_NEED_SECOND_ORDER') & params.op.eq('=~') &
                params.lhs.isin(['STRATEGY','FUNCTIONAL_NEEDS'])].copy()
high['loading_ok']=(pd.to_numeric(high['std.all'],errors='coerce').ge(.50) &
                    pd.to_numeric(high['pvalue'],errors='coerce').lt(.05))
high.to_csv(run_dir/'20_tai_nhan_to_bac_cao.csv',index=False,encoding='utf-8-sig')
first=params.loc[params.model.eq('FULL_STRATEGY_NEED_SECOND_ORDER') & params.op.eq('=~') &
                 params.lhs.isin(SCALE_ITEMS)].copy()
first['loading_ok']=(pd.to_numeric(first['std.all'],errors='coerce').ge(.50) &
                     pd.to_numeric(first['std.all'],errors='coerce').abs().le(1.05))
first.to_csv(run_dir/'20b_tai_nhan_to_bac_mot.csv',index=False,encoding='utf-8-sig')
core=cor.loc[cor.model.eq('FULL_STRATEGY_NEED_SECOND_ORDER') &
             ~cor.latent_1.isin(TIERS['TIER2']+TIERS['TIER3']) &
             ~cor.latent_2.isin(TIERS['TIER2']+TIERS['TIER3'])].copy()
max_core=float(core.correlation.abs().max()) if len(core) else np.nan
requirements=[
 {'dieu_kien':'Combined model convergence, diagnostics, variances, and parameter covariance',
  'dat':bool(target.admissible) and finite(fits.set_index('model').loc['FULL_STRATEGY_NEED_SECOND_ORDER','vcov_eigen_min']),
  'gia_tri':str(target.admissible)},
 {'dieu_kien':'Absolute model fit', 'dat':bool(target.absolute_fit_pass),'gia_tri':target.absolute_fit_pass},
 {'dieu_kien':'Acceptable change relative to 16 correlated scales', 'dat':bool(target.relative_pass),'gia_tri':target.relative_pass},
 {'dieu_kien':'Eight STRATEGY loadings >= 0.50 with p < 0.05',
  'dat':len(high.loc[high.lhs.eq('STRATEGY')])==8 and high.loc[high.lhs.eq('STRATEGY'),'loading_ok'].all(),
  'gia_tri':str(pd.to_numeric(high.loc[high.lhs.eq('STRATEGY'),'std.all'],errors='coerce').min())},
 {'dieu_kien':'Three FUNCTIONAL_NEEDS loadings >= 0.50 with p < 0.05',
  'dat':len(high.loc[high.lhs.eq('FUNCTIONAL_NEEDS')])==3 and high.loc[high.lhs.eq('FUNCTIONAL_NEEDS'),'loading_ok'].all(),
  'gia_tri':str(pd.to_numeric(high.loc[high.lhs.eq('FUNCTIONAL_NEEDS'),'std.all'],errors='coerce').min())},
 {'dieu_kien':'All 71 first-order loadings >= 0.50 and absolute value <= 1.05',
  'dat':len(first)==71 and first.loading_ok.all(),
  'gia_tri':str(pd.to_numeric(first['std.all'],errors='coerce').min()) if len(first) else 'not available'},
 {'dieu_kien':'Core scales are not nearly collinear (|r| < 0.95)', 'dat':finite(max_core) and max_core<LATENT_CORR_CRITICAL,
  'gia_tri':max_core},
]
ready=pd.DataFrame(requirements)
ready.to_csv(run_dir/'21_dieu_kien_san_sang_SEM.csv',index=False,encoding='utf-8-sig')
conclusion=('The combined model meets the stated measurement criteria; proceed to consider structural SEM.'
            if ready.dat.all() else
            'The combined model does not yet meet every criterion for SEM; inspect failed checks and competing models.')
pd.DataFrame([{'ket_luan':conclusion,'tat_ca_dieu_kien_dat':bool(ready.dat.all()),
               'so_dieu_kien_dat':int(ready.dat.sum()),'tong_dieu_kien':len(ready)}]).to_csv(
               run_dir/'22_ket_luan_pha_1.csv',index=False,encoding='utf-8-sig')
display(ready)
print(conclusion)

## 11. Save scores and audit records

Scale means are descriptive candidates and are not latent variable estimates.


In [ ]:
scores=pd.DataFrame({'RESPONDENT_ID':df.RESPONDENT_ID})
for s,items in SCALE_ITEMS.items(): scores[s]=df[items].mean(axis=1,skipna=False)
scores.to_csv(run_dir/'23_diem_thang_ung_vien.csv',index=False,encoding='utf-8-sig')
dictionary=pd.DataFrame([{'thang':s,'items':'|'.join(items),'phuong_phap':'item mean, only when complete',
                          'y_nghia':'descriptive score, not a latent variable'} for s,items in SCALE_ITEMS.items()])
dictionary.to_csv(run_dir/'24_tu_dien_diem_thang.csv',index=False,encoding='utf-8-sig')
if sha256(input_path)!=input_sha or sha256(question_path)!=question_sha:
    raise RuntimeError('Input changed during the run; rerun for a valid audit trail.')
versions={'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,
          'R_and_lavaan':r_version,'seed':SEED,'bootstrap_reps':BOOTSTRAP_REPS,
          'source_csv':str(input_path),'source_sha256':input_sha,'question_file':str(question_path),
          'question_sha256':question_sha,'data_correction_provenance':DATA_CORRECTION_PROVENANCE,
          'tecn_authoritative_text_source':str(question_path),
          'tecn_authoritative_text_sha256':hashlib.sha256(json.dumps(
              TECN_OFFICIAL,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest(),
          'n':len(df),'n_71':len(primary),'n_65':len(full65),'run_utc':stamp,
          'method':'ordered CFA WLSMV via R/lavaan; Python for descriptive and HTMT',
          'claim':'Test whether the hypothesized chain of associations is present; causal claims are not supported.'}
(run_dir/'00_cau_hinh_va_nguon.json').write_text(json.dumps(versions,ensure_ascii=False,indent=2),encoding='utf-8')
note=(f'# Phase 1 report\n\nSample size: {len(df)}; complete on 71 items: {len(primary)}.\n\n'
      f'Conclusion: {conclusion}\n\n'
      'Read `21_dieu_kien_san_sang_SEM.csv`, `18_so_sanh_va_hau_kiem_mo_hinh.csv`, '
      '`19_bang_chung_chin_cap.csv`, and the R log before specifying phase 2. '
      'The item mean scores are not latent variable estimates.\n')
(run_dir/'25_bao_cao_tong_hop.md').write_text(note,encoding='utf-8')
manifest=pd.DataFrame([{'ten_tep':p.name,'bytes':p.stat().st_size,'sha256':sha256(p)}
                       for p in sorted(run_dir.iterdir()) if p.is_file()])
manifest.to_csv(run_dir/'26_manifest_sha256.csv',index=False,encoding='utf-8-sig')
archive=shutil.make_archive(str(run_dir),'zip',root_dir=run_dir)
print('Saved',len(manifest),'result files to:',run_dir,'\nZIP:',archive)

## Interpretation

Inspect validation, complete question text, CFA fit, model comparisons, HTMT, the R log, and the file manifest. Cross sectional associations do not establish causation.
